# QLoRA Training: NJ Housing Price Predictor

Fine-tunes Qwen2.5-0.5B with 4-bit QLoRA on NJ housing data.

## What this notebook does
1. Mounts Google Drive for checkpoint persistence
2. Installs training dependencies (transformers, peft, bitsandbytes, trl)
3. Loads Qwen2.5-0.5B with 4-bit NF4 quantization
4. Applies LoRA adapters (r=8, alpha=16) to q_proj and v_proj
5. Trains with SFTTrainer on JSONL data from Phase 1
6. Saves LoRA adapter checkpoint to Google Drive
7. Plots and saves training loss curve

## Requirements
- Google Colab with GPU runtime (T4 or A100)
- Phase 1 data files uploaded: data/splits/train.jsonl, data/splits/val.jsonl
- lambda/prompt_utils.py uploaded to Colab working directory

## Time Budget
Target: under 20 minutes end-to-end on Colab free tier T4 GPU.

In [ ]:
import os
import time

# Mount Google Drive for checkpoint persistence (survives disconnects)
from google.colab import drive
drive.mount("/content/drive")

# Paths on Google Drive
DRIVE_BASE = "/content/drive/MyDrive/housing_model"
DRIVE_CHECKPOINTS = os.path.join(DRIVE_BASE, "checkpoints")
DRIVE_ADAPTER = os.path.join(DRIVE_BASE, "lora_adapter")
DRIVE_PLOTS = os.path.join(DRIVE_BASE, "plots")

for d in [DRIVE_CHECKPOINTS, DRIVE_ADAPTER, DRIVE_PLOTS]:
    os.makedirs(d, exist_ok=True)

print(f"Drive paths configured:")
print(f"  Checkpoints: {DRIVE_CHECKPOINTS}")
print(f"  Adapter:     {DRIVE_ADAPTER}")
print(f"  Plots:       {DRIVE_PLOTS}")

# Track total time
_start_time = time.time()

In [ ]:
%%capture install_output
!pip install \
    transformers==5.2.0 \
    peft==0.18.1 \
    bitsandbytes==0.49.2 \
    accelerate==1.12.0 \
    trl==0.29.0 \
    datasets==4.6.0 \
    sentencepiece==0.2.1 \
    matplotlib==3.10.8

In [ ]:
# Print install summary (not the full output)
print("Dependencies installed. Key packages:")
import transformers, peft, bitsandbytes, trl, accelerate
print(f"  transformers: {transformers.__version__}")
print(f"  peft:         {peft.__version__}")
print(f"  bitsandbytes: {bitsandbytes.__version__}")
print(f"  trl:          {trl.__version__}")
print(f"  accelerate:   {accelerate.__version__}")

In [ ]:
import json
import sys
import torch
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
    TaskType,
)
from trl import SFTTrainer
from datasets import Dataset

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Option A: Files uploaded to Colab via file browser
# Option B: Files cloned/copied from Drive
# This cell handles both cases.

import importlib

# Try to find lambda/prompt_utils.py
# If running from a cloned repo, it's at ./lambda/prompt_utils.py
# If files uploaded flat, user may need to create the directory structure

_colab_cwd = os.getcwd()
_possible_roots = [
    _colab_cwd,                                              # repo root
    os.path.join(_colab_cwd, "housing_price_predictor"),     # cloned repo
    "/content/housing_price_predictor",                       # common Colab path
    "/content",                                               # flat upload
]

repo_root = None
for candidate in _possible_roots:
    if os.path.isfile(os.path.join(candidate, "lambda", "prompt_utils.py")):
        repo_root = candidate
        break

if repo_root is None:
    print("ERROR: Cannot find lambda/prompt_utils.py")
    print("Please upload the project files to Colab. Options:")
    print("  1. Upload the entire repo via: !git clone <repo_url>")
    print("  2. Upload lambda/prompt_utils.py and data/splits/*.jsonl via file browser")
    print("     Then create lambda/ directory: !mkdir -p lambda && mv prompt_utils.py lambda/")
    raise FileNotFoundError("lambda/prompt_utils.py not found in any expected location")

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import using importlib because 'lambda' is a Python reserved keyword
# Direct 'from lambda.prompt_utils import ...' may raise SyntaxError in some contexts
_prompt_mod = importlib.import_module("lambda.prompt_utils")
format_prompt = _prompt_mod.format_prompt
parse_price_from_output = _prompt_mod.parse_price_from_output

# Verify import
_test = format_prompt(3, 2.0, 1500, 0.25, 1990, "07650", "Single Family")
assert _test.endswith("Predicted price: $"), f"Bad prompt format: {_test!r}"
print(f"Repo root: {repo_root}")
print(f"format_prompt imported and verified.")

# Data paths
TRAIN_JSONL = os.path.join(repo_root, "data", "splits", "train.jsonl")
VAL_JSONL = os.path.join(repo_root, "data", "splits", "val.jsonl")

assert os.path.isfile(TRAIN_JSONL), f"Training data not found: {TRAIN_JSONL}"
assert os.path.isfile(VAL_JSONL), f"Validation data not found: {VAL_JSONL}"

# Count records
with open(TRAIN_JSONL) as f:
    n_train = sum(1 for _ in f)
with open(VAL_JSONL) as f:
    n_val = sum(1 for _ in f)
print(f"Training data:   {n_train:,} records")
print(f"Validation data: {n_val:,} records")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # nested quantization for memory savings
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Set pad token (Qwen2.5 may not have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {model.config.model_type}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Tokenizer vocab size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

In [ ]:
# CRITICAL ORDER: prepare_model_for_kbit_training FIRST, then get_peft_model
# This enables gradient checkpointing on the quantized model and casts
# layer norms to float32 for training stability.
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                          # Low rank -- sufficient for 0.5B regression task
    lora_alpha=16,                # Scaling factor = alpha/r = 2
    lora_dropout=0.05,            # Light regularization
    target_modules=["q_proj", "v_proj"],  # Attention projection layers
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters:     {total_params:,}")
model.print_trainable_parameters()

In [ ]:
def load_jsonl_as_text(jsonl_path: str) -> list:
    """
    Load JSONL file and format each record as a complete training example.

    Each training example is: prompt + price_string
    Example: "Property: Single Family in zip 07650. 3 bedrooms, ... Predicted price: $425000"

    The model learns to generate the price after "Predicted price: $".
    """
    texts = []
    for line in open(jsonl_path, encoding="utf-8"):
        record = json.loads(line)
        prompt = record["prompt"]      # ends with "Predicted price: $"
        price = record["price"]        # float
        # Format price as integer string (no decimals for clean token generation)
        price_str = str(int(round(price)))
        text = prompt + price_str
        texts.append(text)
    return texts


train_texts = load_jsonl_as_text(TRAIN_JSONL)
val_texts = load_jsonl_as_text(VAL_JSONL)

print(f"Training examples:   {len(train_texts):,}")
print(f"Validation examples: {len(val_texts):,}")
print(f"\nSample training text (first 200 chars):")
print(f"  {train_texts[0][:200]}")

# Check token lengths to verify max_seq_length=256 is sufficient
sample_lengths = [len(tokenizer.encode(t)) for t in train_texts[:100]]
print(f"\nToken length stats (first 100 samples):")
print(f"  min={min(sample_lengths)}, max={max(sample_lengths)}, "
      f"mean={np.mean(sample_lengths):.0f}, p95={np.percentile(sample_lengths, 95):.0f}")

if max(sample_lengths) > 256:
    print(f"WARNING: Some samples exceed 256 tokens. Consider increasing max_seq_length.")
else:
    print(f"All samples fit within 256 tokens. max_seq_length=256 is safe.")

# Create HuggingFace Dataset objects
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print(f"\nDatasets created: train={len(train_dataset)}, val={len(val_dataset)}")

In [ ]:
# Training arguments optimized for Colab free tier T4 GPU and 20-min budget
training_args = TrainingArguments(
    output_dir=DRIVE_CHECKPOINTS,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch size = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,                           # T4 supports fp16; use bf16=True on A100
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,                  # keep only last 2 checkpoints (save Drive space)
    gradient_checkpointing=True,         # reduces VRAM at cost of ~10% slower training
    optim="paged_adamw_8bit",            # memory-efficient optimizer for QLoRA
    max_grad_norm=0.3,
    report_to="none",                    # no wandb/tensorboard -- we log manually
    seed=42,
    dataloader_pin_memory=False,         # Colab sometimes has issues with pinned memory
)

# SFTTrainer handles tokenization and formatting for us
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=256,
    dataset_text_field="text",
    packing=False,                       # no packing -- each example is one training sample
)

print(f"Trainer configured.")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size} (effective: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps})")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"  FP16: {training_args.fp16}")
print(f"  Save to: {DRIVE_CHECKPOINTS}")

# Estimate training steps
n_steps = (len(train_dataset) // training_args.per_device_train_batch_size // training_args.gradient_accumulation_steps) * int(training_args.num_train_epochs)
print(f"  Estimated total steps: ~{n_steps}")

In [ ]:
print("Starting training...")
train_start = time.time()

train_result = trainer.train()

train_elapsed = time.time() - train_start
print(f"\nTraining complete in {train_elapsed/60:.1f} minutes")
print(f"  Final training loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

if train_elapsed > 1200:  # 20 minutes
    print(f"WARNING: Training took {train_elapsed/60:.1f} min > 20 min budget (TRAIN-02).")
    print("Consider: reduce num_train_epochs to 2, or reduce dataset size.")
else:
    print(f"TRAIN-02 PASSED: Training completed within 20-minute budget.")

In [ ]:
# Save the LoRA adapter (NOT the full model -- adapter is ~10MB vs ~1GB for full model)
print(f"Saving LoRA adapter to {DRIVE_ADAPTER}...")
model.save_pretrained(DRIVE_ADAPTER)
tokenizer.save_pretrained(DRIVE_ADAPTER)

# List saved files
saved_files = os.listdir(DRIVE_ADAPTER)
print(f"Saved files ({len(saved_files)}):")
for f in sorted(saved_files):
    fpath = os.path.join(DRIVE_ADAPTER, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

# Verify adapter_config.json exists (required for PeftModel.from_pretrained)
assert "adapter_config.json" in saved_files, "adapter_config.json missing -- save failed"
print(f"\nAdapter saved successfully to Google Drive.")
print(f"Path: {DRIVE_ADAPTER}")

In [ ]:
# Quick verification: reload adapter from Drive and run one inference
# This confirms the saved adapter is valid and can be used in a fresh session
print("Verifying adapter reload...")

from peft import PeftModel

# Reload base model (still quantized for this quick check)
base_for_check = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load saved adapter
reloaded_model = PeftModel.from_pretrained(base_for_check, DRIVE_ADAPTER)
print("Adapter reloaded successfully from Drive.")

# Quick inference test
test_prompt = format_prompt(3, 2.0, 1800, 0.25, 1995, "07650", "Single Family")
inputs = tokenizer(test_prompt, return_tensors="pt").to(reloaded_model.device)

with torch.no_grad():
    outputs = reloaded_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        temperature=1.0,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the generated part (after the prompt)
generated_price_text = generated[len(test_prompt):]
parsed_price = parse_price_from_output(generated_price_text)

print(f"\nTest inference:")
print(f"  Prompt: ...{test_prompt[-60:]}")
print(f"  Generated: {generated_price_text[:50]}")
print(f"  Parsed price: {parsed_price}")

if parsed_price is not None and 50_000 < parsed_price < 5_000_000:
    print("Inference check PASSED: Generated a plausible NJ housing price.")
else:
    print("WARNING: Generated price may be implausible. Review training quality.")
    print("This is acceptable for initial training -- model accuracy improves with better data (DATA-04).")

# Clean up to free VRAM
del base_for_check, reloaded_model
torch.cuda.empty_cache()

In [ ]:
# Extract training and eval loss from trainer log history
train_losses = [(entry["step"], entry["loss"]) for entry in trainer.state.log_history if "loss" in entry]
eval_losses = [(entry["step"], entry["eval_loss"]) for entry in trainer.state.log_history if "eval_loss" in entry]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Training Loss", color="steelblue", alpha=0.8)

if eval_losses:
    eval_steps, eval_loss_vals = zip(*eval_losses)
    ax.plot(eval_steps, eval_loss_vals, label="Validation Loss", color="coral", alpha=0.8, marker="o", markersize=4)

ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("QLoRA Training Loss -- Qwen2.5-0.5B on NJ Housing Data")
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate final loss
if train_losses:
    final_step, final_loss = train_losses[-1]
    ax.annotate(f"Final: {final_loss:.4f}", xy=(final_step, final_loss),
                xytext=(-60, 20), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color="gray"),
                fontsize=9, color="steelblue")

plt.tight_layout()
loss_plot_path = os.path.join(DRIVE_PLOTS, "training_loss_curve.png")
plt.savefig(loss_plot_path, dpi=150)
plt.show()
plt.close()

print(f"Loss curve saved to: {loss_plot_path}")

# Check that loss decreased (success criterion 3)
if len(train_losses) >= 2:
    first_loss = train_losses[0][1]
    last_loss = train_losses[-1][1]
    if last_loss < first_loss:
        print(f"Loss decreased: {first_loss:.4f} -> {last_loss:.4f} (PASSED)")
    else:
        print(f"WARNING: Loss did not decrease: {first_loss:.4f} -> {last_loss:.4f}")
        print("Consider: lower learning rate, more epochs, or check data quality.")

In [ ]:
total_elapsed = time.time() - _start_time

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model:           {MODEL_ID}")
print(f"LoRA rank:       {lora_config.r}")
print(f"LoRA alpha:      {lora_config.lora_alpha}")
print(f"Target modules:  {lora_config.target_modules}")
print(f"Training records: {len(train_texts):,}")
print(f"Epochs:          {training_args.num_train_epochs}")
print(f"Effective batch:  {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate:   {training_args.learning_rate}")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"Training time:   {train_elapsed/60:.1f} min")
print(f"Total time:      {total_elapsed/60:.1f} min (includes setup + installs)")
print(f"Adapter saved:   {DRIVE_ADAPTER}")
print(f"Loss plot:       {loss_plot_path}")
print("=" * 60)

# Final budget check
if total_elapsed <= 1200:
    print(f"\nTRAIN-02 PASSED: Total time {total_elapsed/60:.1f} min <= 20 min budget.")
else:
    print(f"\nTRAIN-02 WARNING: Total time {total_elapsed/60:.1f} min > 20 min budget.")
    print("Optimization suggestions:")
    print("  - Reduce num_train_epochs to 2")
    print("  - Reduce dataset to 3,000 records")
    print("  - Use A100 GPU if available (Runtime > Change runtime type)")